# 04 - Defense Evaluation

Apply Input Smoothing and Adversarial Training defenses to robustify EEGNet.

In [ ]:
import sys
sys.path.append('..')
import copy
import torch
import torch.nn as nn
from src.models.eegnet import EEGNet
from src.data.dataset import get_dataloaders
from src.defenses.input_smoothing import GaussianSmoothing
from src.defenses.adversarial_training import train_adversarial_epoch
from experiments.run_experiment import train_base_model, evaluate_model
from src.attacks.pgd import pgd_attack

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader, test_loader = get_dataloaders('../data/raw/', 1, batch_size=32)

# 1. Base Model & Attack
model = EEGNet().to(device)
model = train_base_model(model, train_loader, epochs=10, lr=0.001, device=device)
acc_adv = evaluate_model(model, test_loader, device=device, attack_fn=pgd_attack, epsilon=0.1)
print(f"No Defense PGD acc: {acc_adv:.2f}%")

# 2. Input Smoothing
smoothing = GaussianSmoothing(channels=22, kernel_size=5, sigma=1.0).to(device)
acc_smooth = evaluate_model(model, test_loader, device=device, attack_fn=pgd_attack, epsilon=0.1, defense_module=smoothing)
print(f"Input Smoothing PGD acc: {acc_smooth:.2f}%")

# 3. Adversarial Training
adv_model = copy.deepcopy(model).to(device)
optimizer = torch.optim.Adam(adv_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
print("Adversarial Training...")
for _ in range(5):
    train_adversarial_epoch(adv_model, train_loader, optimizer, criterion, pgd_attack, device, epsilon=0.1, alpha=0.5)

adv_train_acc = evaluate_model(adv_model, test_loader, device=device, attack_fn=pgd_attack, epsilon=0.1)
print(f"Adversarial Training PGD acc: {adv_train_acc:.2f}%")
